# ML-11 — Capstone: Refresh / Content Opportunity Scoring

**Author:** Albert Michael Tigo  
**Lane:** Lane 2 — Refresh / Content Opportunity Scoring

This notebook consolidates the capstone question, data contract, baseline, model comparison, validation audit, action playbook, reproducibility checks and ML-12 communication outputs. Run it from the repository root in Colab or locally.

**Important:** the starter snapshot is cross-sectional. The 90-day features overlap the 30-day label window, so this is contemporaneous ranking/decision support—not a future-decline forecast.

## 1. Research question

Can observable content and search signals help an editor prioritize pages for refresh review? The unit is one pseudonymous content item. The output is a ranked queue with a score, reason codes and suggested action. A human editor verifies every recommendation; the model never publishes or rewrites content.

## 2. Data and safety contract

Use the anonymized starter CSV (30,000 rows × 44 columns; 32 pseudonymous clients). The reference target is `is_declining_label`, derived from `trend_direction == 'down'` (16,262 positives; 54.2%). IDs are for grouping only. Exclude `trend_direction` and `trend_pct` from features because they define/encode the target; exclude IDs and product-decision flags. Do not expose client names, domains, URLs, titles or raw queries.

**Temporal caveat:** 90-day totals overlap the 30-day period used to construct the label. This snapshot cannot support a clean future prediction claim.

In [40]:
import json

## 3. Baseline

The ML-07 rule flags pages at least 180 days since update and with at least 500 impressions in the 90-day window. Score = stale × visible × impressions. Reason code: `stale_and_visible`; action: `review_refresh`. The saved baseline notebook reports 17 actionable pages out of 30,000.

In [41]:
import os
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/alberttigo24-droid/Albert-Flyrank-assignments.git"
REPO_DIR = Path("/content/Albert-Flyrank-assignments")

if REPO_DIR.exists():
    subprocess.run(
        ["git", "pull", "origin", "main"],
        cwd=REPO_DIR,
        check=True
    )
else:
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True
    )

os.chdir(REPO_DIR)
ROOT = Path.cwd()

print("Repository root:", ROOT)
print("Python:", sys.version)

assert (ROOT / "scripts" / "run_all.py").exists()

print("Repository setup successful!")

Repository root: /content/Albert-Flyrank-assignments
Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Repository setup successful!


In [42]:
subprocess.check_call([
    sys.executable, "-m", "pip",
    "install", "-r", "requirements.txt"
])

0

## 4. Re-run the reference pipeline

This regenerates the baseline, model metrics, predictions, ranked queue and model report from the anonymized starter dataset. It may take a few minutes.

In [43]:

result = subprocess.run(
    [sys.executable, "scripts/run_all.py"],
    cwd=ROOT,
    text=True,
    capture_output=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Pipeline execution failed")

print("Pipeline completed successfully!")



▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/Albert-Flyrank-assignments/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/Albert-Flyrank-assignments/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/Albert-Flyrank-assignments/data/processed/model_predictions.csv
Wrote model results: /content/Albert-Flyrank-assignments/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /content/Albert-Flyrank-assignments/outputs/refresh_queue.csv
Wrote mo

## 5. Results (same held-out split)

Load the regenerated metrics. The target base rate must be shown next to ranking metrics. The reference report's saved headline values are included as a cross-check, not as a substitute for rerunning.

In [44]:
metrics_path = ROOT / 'outputs' / 'model_results.json'
report_path = ROOT / 'outputs' / 'model_report.md'
if not metrics_path.exists():
    raise FileNotFoundError('Expected outputs/model_results.json after running the pipeline.')
metrics = json.loads(metrics_path.read_text())
print('Rows:', metrics['input_rows'])
print('Split:', metrics['split_strategy'])
print('Positive rate:', round(metrics['target_positive_rate'], 3))
print('Best model:', metrics['best_model']['name'])
print('\nModel comparison')
for name, m in metrics['models'].items():
    print(name, 'ROC AUC=', round(m['roc_auc'],3), 'AP=', round(m['average_precision'],3), 'P@50=', round(m['precision_at_50'],3), 'Recall=', round(m['recall'],3), 'F1=', round(m['f1'],3))
print('Baseline:', metrics['baseline'])
print('\nTop features:')
for item in metrics['best_model']['feature_importance_top'][:10]: print(item)

Rows: 30000
Split: client_holdout
Positive rate: 0.542
Best model: random_forest

Model comparison
decision_tree ROC AUC= 0.742 AP= 0.575 P@50= 0.58 Recall= 0.716 F1= 0.634
logistic_regression ROC AUC= 0.7 AP= 0.522 P@50= 0.4 Recall= 0.567 F1= 0.566
random_forest ROC AUC= 0.75 AP= 0.618 P@50= 0.74 Recall= 0.744 F1= 0.64
Baseline: {'baseline_accuracy': 0.6086021505376344, 'baseline_average_precision': 0.46760658636934826, 'baseline_f1': 0.2743221690590112, 'baseline_precision': 0.4985507246376812, 'baseline_precision_at_100': 0.36, 'baseline_precision_at_20': 0.15, 'baseline_precision_at_50': 0.24, 'baseline_recall': 0.18921892189218922, 'baseline_roc_auc': 0.6268917852237201}

Top features:
{'feature': 'days_with_impressions', 'importance': 0.15814381552310994}
{'feature': 'log_impressions_90d', 'importance': 0.12863800983285106}
{'feature': 'avg_position', 'importance': 0.10916404306325882}
{'feature': 'content_age_days', 'importance': 0.0951682857931063}
{'feature': 'char_count', 'im

## 6. Leakage and validation audit

The reference script attempts a client holdout (20% of clients) and falls back to a stratified row split if the held-out client set lacks both classes. Confirm which split was actually used. The model feature list must not contain `trend_direction`, `trend_pct`, `is_declining_label`, `client_id` or `content_id`.

The more important caveat is temporal: the label is derived from last-30-day versus previous-30-day impressions, while several model inputs aggregate the same trailing 90-day period. This overlap means the current result is not a future forecast. A future-facing model needs features strictly before the label window using daily warehouse data.

In [45]:
from scripts import ml_utils
forbidden = {'trend_direction','trend_pct','is_declining_label','client_id','content_id'}
configured = set(ml_utils.MODEL_NUMERIC_FEATURES) | set(ml_utils.MODEL_CATEGORICAL_FEATURES)
print('Forbidden feature overlap:', sorted(forbidden & configured))
assert not (forbidden & configured), 'Leakage/identifier field found in configured features.'
print('Configured feature count:', len(configured))
print('Split strategy:', metrics['split_strategy'])
print('NOTE: no temporal separation is possible from this single 90-day snapshot.')

Forbidden feature overlap: []
Configured feature count: 26
Split strategy: client_holdout
NOTE: no temporal separation is possible from this single 90-day snapshot.


## 7. Ranked recommendations and human review

Use `outputs/refresh_queue.csv` as a triage queue. Action labels map to manual checks:

- `refresh_and_review_ctr`: inspect snippet/title alignment and search intent.
- `refresh_and_review_engagement`: inspect usefulness, page experience and answer quality.
- `refresh`: check for substantive content gaps before editing.
- `expand_and_refresh`: verify whether additional coverage is warranted.
- `monitor`: defer immediate work and monitor signals.

Never infer real page identity from pseudonymous IDs. Never auto-publish, delete or rewrite from a score alone.

In [46]:
queue_path = ROOT / 'outputs' / 'refresh_queue.csv'
import pandas as pd
queue = pd.read_csv(queue_path)
print('Queue rows:', len(queue))
print('Action counts:')
print(queue['suggested_action'].value_counts().to_string())
print('\nTop 10 review rows (pseudonymous IDs only):')
cols = [c for c in ['final_rank','final_refresh_score','confidence','suggested_action','final_reason_codes','impressions_90d','sessions_90d'] if c in queue.columns]
display(queue[cols].head(10))

Queue rows: 30000
Action counts:
suggested_action
monitor                          13083
refresh                           8188
refresh_and_review_ctr            6654
refresh_and_review_engagement     1993
expand_and_refresh                  82

Top 10 review rows (pseudonymous IDs only):


,final_rank,final_refresh_score,confidence,suggested_action,final_reason_codes,impressions_90d,sessions_90d
0,1,81.734212,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,12834,66
1,2,81.603243,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,2498,9
2,3,81.544618,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,8064,23
3,4,81.169731,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,13790,27
4,5,80.957565,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,3393,5
5,6,80.798090,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,5811,14
6,7,80.650656,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,1622,20
7,8,80.432641,medium,refresh,declining_with_demand|model_decline_risk|visib...,4366,8
8,9,80.428403,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,1597,5
9,10,80.428243,medium,refresh,declining_with_demand|model_decline_risk|visib...,3867,5


## 8. Limitations and reproducibility

This target is a trend proxy, not a refresh-success label. The data is cross-sectional, the feature windows overlap the target window, and client holdout is not time holdout. The result is not causal and should not be described as predicting Google's ranking algorithm. The reference report's headline metrics are historical saved outputs; rerun the pipeline above to regenerate the metrics JSON and predictions.

**Re-run:** `pip install -r requirements.txt` then `python scripts/run_all.py`. Random seed: 42. Keep raw datasets and generated per-page queues out of Git.

## 9. Acknowledgments and data credit

Built on the FlyRank ML Internship anonymized starter dataset and teaching pipeline. Data credit: [FlyRank](https://flyrank.ai).

## ML-12 — Five-minute demo outline

1. 0:00–0:40 — editorial triage problem.
2. 0:40–1:20 — anonymized 30,000-row snapshot and data safety.
3. 1:20–2:10 — stale-and-visible baseline.
4. 2:10–3:20 — model comparison, client holdout and temporal-overlap caveat.
5. 3:20–4:20 — ranked queue, reason codes and human review.
6. 4:20–5:00 — limitations and next step: time-separated validation using daily warehouse data.

### Social-post cut

I built a content-refresh prioritization experiment using FlyRank's anonymized search dataset (30,000 pages). The saved evaluation report records Random Forest Precision@50 of 0.74 versus 0.24 for the rule baseline. The key lesson: time-window alignment matters. This single-snapshot result is decision support—not proof that refreshing content causes traffic recovery or a future-ranking forecast.

### Employer-facing summary

I built an explainable content-refresh prioritization pipeline on a 30,000-row anonymized search dataset, including a rule baseline, three classifiers, client-holdout evaluation, reason codes and a ranked editorial review queue. The committed pipeline report records Random Forest ROC AUC 0.750 and Precision@50 0.740, compared with baseline Precision@50 0.240. I documented temporal overlap and framed the output as human-reviewed decision support rather than a future-performance or causal claim.

In [47]:
from pathlib import Path
import json

output_dir = ROOT / "outputs"

files = [
    "model_results.json",
    "model_report.md",
    "refresh_queue.csv",
    "summary.json",
    "flyrank_refresh_model_results.pdf"
]

for name in files:
    path = output_dir / name
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}")

pdf_path = output_dir / "flyrank_refresh_model_results.pdf"
if pdf_path.exists():
    print("PDF size:", round(pdf_path.stat().st_size / 1024, 2), "KB")

model_results.json: FOUND
model_report.md: FOUND
refresh_queue.csv: FOUND
summary.json: FOUND
flyrank_refresh_model_results.pdf: FOUND
PDF size: 17.09 KB


In [48]:
import json

for filename in ["model_results.json", "summary.json"]:
    path = ROOT / "outputs" / filename
    data = json.loads(path.read_text())
    print(f"\n--- {filename} ---")
    print(json.dumps(data, indent=2)[:10000])


--- model_results.json ---
{
  "baseline": {
    "baseline_accuracy": 0.6086021505376344,
    "baseline_average_precision": 0.46760658636934826,
    "baseline_f1": 0.2743221690590112,
    "baseline_precision": 0.4985507246376812,
    "baseline_precision_at_100": 0.36,
    "baseline_precision_at_20": 0.15,
    "baseline_precision_at_50": 0.24,
    "baseline_recall": 0.18921892189218922,
    "baseline_roc_auc": 0.6268917852237201
  },
  "best_model": {
    "feature_importance_top": [
      {
        "feature": "days_with_impressions",
        "importance": 0.15814381552310994
      },
      {
        "feature": "log_impressions_90d",
        "importance": 0.12863800983285106
      },
      {
        "feature": "avg_position",
        "importance": 0.10916404306325882
      },
      {
        "feature": "content_age_days",
        "importance": 0.0951682857931063
      },
      {
        "feature": "char_count",
        "importance": 0.042608309447693275
      },
      {
        "feature

In [49]:
report_path = ROOT / "outputs" / "model_report.md"
print(report_path.read_text())

# FlyRank Refresh Opportunity Model Report

This report is generated from the bundled anonymized starter dataset (`data/raw/content_refresh_anonymized.csv`).
The model ranks existing content for refresh review. It does not use titles, URLs, client names, domains, or keywords.

## Data

- Rows scored: 30,000
- Declining-label rows: 16,262
- Declining-label rate: 0.542
- Split strategy used for validation: client_holdout
- Target: `is_declining_label`

## Model Comparison

Best model: `random_forest` selected by `precision_at_50`.

| Model | ROC AUC | Avg precision | Precision@50 | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| decision_tree | 0.742 | 0.575 | 0.580 | 0.716 | 0.634 |
| logistic_regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
| baseline_rules | 0.627 | 0.468 | 0.240 | - | - |

## Final Queue

- High-confidence items: 3,602
- Medium-confidence items: 11,398
- Low-confidence items: 15,000
- `monitor` items: 13,08